In [1]:
%pip install numpy pandas librosa torch torchaudio matplotlib seaborn tqdm

Note: you may need to restart the kernel to use updated packages.


In [27]:
%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 25.5 MB/s  0:00:14:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 19.5 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 40.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 17.0 MB/s  0:00:01m0:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 12.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 25.4 MB/s  0:00:00m0:00:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 17.9 MB/s  0:00:26:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 22.0 MB/s  0:00:16:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 24.6 MB/s  0:00:04:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 23.2 MB/s  0:00:02m0:00:0100

In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import torchaudio
import warnings
import logging
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

/home-mscluster/kaspoas/miniconda3/envs/acml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

2.5.1+cu121
12.1
True
2


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Dataset location on the cluster
# ─────────────────────────────────────────────────────────────────────────────
# Large datasets should be stored in /datasets/<username>/ or /gluster/<username>/,
# not in your home directory.
#
# Expected structure:
# /datasets/klaspoas/fma/
# ├── fma_metadata/
# │   └── tracks.csv
# ├── fma_small/
# │   ├── 000/
# │   ├── 001/
# │   └── ...
# └── mel_spectrograms/
#
# You can override this without editing the notebook by running:
# export FMA_DATA_ROOT=/path/to/fma
DATA_ROOT = os.environ.get("FMA_DATA_ROOT", "/datasets/kaspoas")

METADATA_DIR = os.path.join(DATA_ROOT, "fma_metadata")
AUDIO_DIR = os.path.join(DATA_ROOT, "fma_small")
SPEC_DIR = os.path.join(DATA_ROOT, "mel_spectrograms")

os.makedirs(SPEC_DIR, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("METADATA_DIR:", METADATA_DIR)
print("AUDIO_DIR:", AUDIO_DIR)
print("SPEC_DIR:", SPEC_DIR)

DATA_ROOT: /datasets/kaspoas
METADATA_DIR: /datasets/kaspoas/fma_metadata
AUDIO_DIR: /datasets/kaspoas/fma_small
SPEC_DIR: /datasets/kaspoas/mel_spectrograms


In [4]:

#defaults
DURATION = 30                   
SR = 22050        #sample rate
N_MELS = 64      #num frequency bins
N_FFT = 2048      #frequency resolution - for Fourier Transform
HOP_LEN = 512     #time resolution - step size between windows

MAX_FRAMES  = 650                  # pad/truncate to this many time frames
NUM_CLASSES = 8
BATCH_SIZE  = 32
NUM_EPOCHS  = 30
LR          = 1e-3

logging.getLogger("torchaudio").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

### Load Data

In [5]:
#returns Dataframe with (filepath, genre)
def load_fma_metadata():
    tracks = pd.read_csv(
        os.path.join(METADATA_DIR, "tracks.csv"),
        index_col=0,
        header=[0, 1]
    )

    small = tracks[tracks[("set", "subset")] == "small"]
    genres = small[("track", "genre_top")].dropna()
    return genres

def track_id_to_path(track_id):
    tid = f"{int(track_id):06d}"
    return os.path.join(AUDIO_DIR, tid[:3], f"{tid}.mp3")

#Load an audio file and return a log-mel spectrogram as a numpy array
"""
def audio_to_melspec(path):
    try:
        y, sr = librosa.load(path)
    except Exception:
        return np.zeros((N_MELS, MAX_FRAMES), dtype=np.float32)

    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
    log_mel = librosa.power_to_db(mel_spec, ref=np.max).astype(np.float32)

    # Pad or truncate time axis
    if log_mel.shape[1] < MAX_FRAMES:
        log_mel = np.pad(log_mel, ((0, 0), (0, MAX_FRAMES - log_mel.shape[1])))
    else:
        log_mel = log_mel[:, :MAX_FRAMES]

    # Normalise to [0, 1]
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-6)
    return log_mel
"""

mel_transform = torchaudio.transforms.MelSpectrogram(sample_rate=SR,
    n_fft=N_FFT,
    hop_length=HOP_LEN,
    n_mels=N_MELS
)

#torchaudio
def audio_to_melspec(path):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            waveform, native_sr = torchaudio.load(path)

        # Convert to mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample if needed
        if native_sr != SR:
            waveform = torchaudio.transforms.Resample(native_sr, SR)(waveform)

        # Trim to duration
        max_samples = SR * DURATION
        waveform = waveform[:, :max_samples]

        # Return zeros if too short (corrupt file)
        if waveform.shape[1] < SR * 2:
            return np.zeros((N_MELS, MAX_FRAMES), dtype=np.float32)
        
        mel = mel_transform(waveform).squeeze(0).numpy()

        # Log scale
        log_mel = 10.0 * np.log10(np.maximum(mel, 1e-10)).astype(np.float32)

    except Exception:
        return np.zeros((N_MELS, MAX_FRAMES), dtype=np.float32)

    # Pad or truncate
    if log_mel.shape[1] < MAX_FRAMES:
        log_mel = np.pad(log_mel, ((0, 0), (0, MAX_FRAMES - log_mel.shape[1])))
    else:
        log_mel = log_mel[:, :MAX_FRAMES]

    # Normalise
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-6)
    return log_mel

def precompute_spectrograms(track_ids):
    os.makedirs(SPEC_DIR, exist_ok=True)
    print(f"Precomputing spectrograms for {len(track_ids)} tracks...")
    for i, tid in enumerate(track_ids):
        cache_path = os.path.join(SPEC_DIR, f"{tid}.npy")
        if os.path.exists(cache_path):
            continue  # already computed, skip
        path = track_id_to_path(tid)
        spec = audio_to_melspec(path)
        np.save(cache_path, spec)
        if i % 100 == 0:
            print(f"  {i}/{len(track_ids)} done")
    print("Precomputation complete.")

#dataset wih melspectogram and label
class FMADataset(Dataset):
    
    def __init__(self, track_ids, labels, transform=None):
        self.track_ids = track_ids
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.track_ids)

    def __getitem__(self, idx):
        cache_path = os.path.join(SPEC_DIR, f"{self.track_ids[idx]}.npy")
        spec  = np.load(cache_path)
        spec  = torch.from_numpy(spec).unsqueeze(0)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return spec, label


In [6]:
def check_environment_and_data(train_ids=None, val_ids=None, test_ids=None, class_names=None, n_examples=5):
    """
    Print checks to confirm that:
    1. GPU/PyTorch can see CUDA.
    2. FMA metadata exists and has the expected columns.
    3. Audio files exist at the expected paths.
    4. Cached spectrogram .npy files exist.
    5. Spectrograms can be loaded and have the expected shape.
    """
    print("\n" + "=" * 70)
    print("ENVIRONMENT CHECK")
    print("=" * 70)
    print("Python working directory:", os.getcwd())
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA device count:", torch.cuda.device_count())

    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    else:
        print("WARNING: CUDA is not available. This job is running on CPU.")

    print("\n" + "=" * 70)
    print("FMA FILE CHECK")
    print("=" * 70)

    tracks_csv = os.path.join(METADATA_DIR, "tracks.csv")
    print("DATA_ROOT:", DATA_ROOT)
    print("tracks.csv path:", tracks_csv)
    print("tracks.csv exists:", os.path.exists(tracks_csv))

    print("fma_small directory exists:", os.path.isdir(AUDIO_DIR))
    print("SPEC_DIR:", SPEC_DIR)
    print("mel_spectrograms directory exists:", os.path.isdir(SPEC_DIR))

    if not os.path.exists(tracks_csv):
        print("ERROR: fma_metadata/tracks.csv was not found from the current working directory.")
        return

    tracks = pd.read_csv(tracks_csv, index_col=0, header=[0, 1])
    print("tracks.csv shape:", tracks.shape)
    print("First 10 metadata columns:", list(tracks.columns[:10]))

    required_cols = [("set", "subset"), ("set", "split"), ("track", "genre_top")]
    for col in required_cols:
        print(f"Required column {col} exists:", col in tracks.columns)

    small = tracks[tracks[("set", "subset")] == "small"]
    print("FMA small rows:", len(small))
    print("Rows with genre_top:", small[("track", "genre_top")].notna().sum())
    print("Split counts:")
    print(small[("set", "split")].value_counts(dropna=False))
    print("Genre counts:")
    print(small[("track", "genre_top")].value_counts(dropna=False))

    if class_names is not None:
        print("Encoded class names:", class_names)
        print("Number of encoded classes:", len(class_names))

    if train_ids is None:
        genre_series = load_fma_metadata()
        le = LabelEncoder()
        labels = le.fit_transform(genre_series.values)
        class_names = le.classes_.tolist()
        small = small.loc[small.index.isin(genre_series.index)].copy()
        small["label"] = le.transform(small[("track", "genre_top")])
        small["split"] = small[("set", "split")]
        train_ids = small[small["split"] == "training"].index.tolist()
        val_ids = small[small["split"] == "validation"].index.tolist()
        test_ids = small[small["split"] == "test"].index.tolist()

    all_ids = list(train_ids) + list(val_ids) + list(test_ids)
    print("\nDataset split sizes:")
    print("Train:", len(train_ids), "Val:", len(val_ids), "Test:", len(test_ids), "Total:", len(all_ids))

    print("\n" + "=" * 70)
    print("AUDIO + SPECTROGRAM SAMPLE CHECK")
    print("=" * 70)

    sample_ids = all_ids[:n_examples]
    for tid in sample_ids:
        audio_path = track_id_to_path(tid)
        cache_path = os.path.join(SPEC_DIR, f"{tid}.npy")

        print(f"\nTrack ID: {tid}")
        print("  Audio path:", audio_path)
        print("  Audio exists:", os.path.exists(audio_path))
        print("  Spectrogram path:", cache_path)
        print("  Spectrogram exists:", os.path.exists(cache_path))

        if os.path.exists(cache_path):
            try:
                spec = np.load(cache_path)
                print("  Spectrogram shape:", spec.shape)
                print("  Spectrogram dtype:", spec.dtype)
                print("  Spectrogram min/max:", float(np.min(spec)), float(np.max(spec)))
                print("  All finite:", bool(np.isfinite(spec).all()))
                print("  Correct shape:", spec.shape == (N_MELS, MAX_FRAMES))
            except Exception as e:
                print("  ERROR loading spectrogram:", repr(e))
        else:
            print("  WARNING: cached spectrogram missing. Run precompute_spectrograms(all_ids).")

    print("\n" + "=" * 70)
    print("END CHECK")
    print("=" * 70 + "\n")


def check_dataloader_batch(loader, split_name, device=None):
    """
    Print one DataLoader batch shape to confirm that training will receive:
    specs: [batch, 1, N_MELS, MAX_FRAMES]
    labels: [batch]
    """
    print("\n" + "=" * 70)
    print(f"DATALOADER CHECK: {split_name}")
    print("=" * 70)

    specs, labels = next(iter(loader))
    print("Specs batch shape:", tuple(specs.shape))
    print("Labels batch shape:", tuple(labels.shape))
    print("Specs dtype:", specs.dtype)
    print("Labels dtype:", labels.dtype)
    print("Specs min/max:", float(specs.min()), float(specs.max()))
    print("Unique labels in batch:", sorted(labels.unique().tolist()))
    print("Expected specs shape: [batch_size, 1,", N_MELS, ",", MAX_FRAMES, "]")

    if device is not None:
        specs = specs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        print("Moved batch to device:", device)
        print("Specs device:", specs.device)
        print("Labels device:", labels.device)

### Model

In [7]:
class ConvBlock(nn.Module):
    """Conv2D → BatchNorm → ReLU → MaxPool block."""
    def __init__(self, in_ch, out_ch, pool=(2, 2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(pool),
            nn.Dropout2d(0.2),
        )

    def forward(self, x):
        return self.block(x)


class CRNN(nn.Module):
    def __init__(self, lstm_hidden=128, lstm_layers=2, dropout=0.3):
        super().__init__()

        self.cnn = nn.Sequential(
            ConvBlock(1,   16, pool=(2, 2)),
            ConvBlock(16,  32, pool=(2, 2)),
            ConvBlock(32, 64, pool=(4, 1)),
        )

        self.freq_out = N_MELS // (2 * 2 * 4)
        self.lstm_in  = 64 * self.freq_out

        self.lstm = nn.LSTM(
            input_size=self.lstm_in,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden * 2, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, NUM_CLASSES),
        )

    def forward(self, x):
        x = self.cnn(x)
        B, C, F, T = x.shape
        x = x.permute(0, 3, 1, 2).reshape(B, T, C * F)
        x, _ = self.lstm(x)
        return self.classifier(x[:, -1, :])


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for specs, labels in tqdm(loader, desc="  Train", leave=False):
        specs, labels = specs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(specs)
        loss    = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        preds       = outputs.detach().argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for specs, labels in tqdm(loader, desc="  Eval ", leave=False):
        specs, labels = specs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        outputs    = model(specs)
        loss       = criterion(outputs, labels)
        total_loss += loss.item() * labels.size(0)
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels


### Main

In [9]:
def main():
    # ── Device setup ───────────────────────────────────────────────────────
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_gpus = torch.cuda.device_count()
    print(f"Using device: {device}  ({n_gpus} GPU(s))")

    # ── Metadata ───────────────────────────────────────────────────────────
    check_environment_and_data()
    print("Loading FMA metadata...")
    genre_series = load_fma_metadata()

    le = LabelEncoder()
    le.fit_transform(genre_series.values)
    class_names = le.classes_.tolist()
    print(f"Classes found: {len(class_names)} -> {class_names}")
    if len(class_names) != NUM_CLASSES:
        raise ValueError(f"NUM_CLASSES={NUM_CLASSES}, but metadata contains {len(class_names)} classes")

    tracks = pd.read_csv(
        os.path.join(METADATA_DIR, "tracks.csv"),
        index_col=0, header=[0, 1]
    )
    small          = tracks[tracks[("set", "subset")] == "small"].copy()
    small          = small.loc[small.index.isin(genre_series.index)]
    small["label"] = le.transform(small[("track", "genre_top")])
    small["split"] = small[("set", "split")]

    def split_data(split_name):
        subset = small[small["split"] == split_name]
        return subset.index.tolist(), subset["label"].tolist()

    train_ids, train_labels = split_data("training")
    val_ids,   val_labels   = split_data("validation")
    test_ids,  test_labels  = split_data("test")
    print(f"Train: {len(train_ids)}  Val: {len(val_ids)}  Test: {len(test_ids)}")

    all_ids = train_ids + val_ids + test_ids
    check_environment_and_data(train_ids, val_ids, test_ids, class_names)

    missing_specs = [tid for tid in all_ids if not os.path.exists(os.path.join(SPEC_DIR, f"{tid}.npy"))]
    print(f"Missing cached spectrograms: {len(missing_specs)} / {len(all_ids)}")
    if missing_specs:
        print("Precomputing missing spectrograms now...")
        precompute_spectrograms(missing_specs)
        print("Rechecking data after precomputation...")
        check_environment_and_data(train_ids, val_ids, test_ids, class_names)

    # ── DataLoaders ────────────────────────────────────────────────────────
    num_workers = min(4 * max(n_gpus, 1), 4)
    pin         = device.type == "cuda"

    train_loader = DataLoader(FMADataset(train_ids, train_labels), batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=num_workers, pin_memory=pin,
                              persistent_workers=num_workers > 0)
    val_loader   = DataLoader(FMADataset(val_ids,   val_labels),   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=num_workers, pin_memory=pin,
                              persistent_workers=num_workers > 0)
    test_loader  = DataLoader(FMADataset(test_ids,  test_labels),  batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=num_workers, pin_memory=pin,
                              persistent_workers=num_workers > 0)

    check_dataloader_batch(train_loader, "train", device)
    check_dataloader_batch(val_loader, "validation", device)
    check_dataloader_batch(test_loader, "test", device)

    # ── Model ──────────────────────────────────────────────────────────────
    model = CRNN()
    if device.type == "cuda" and torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

    print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(model)

    # ── Training loop ──────────────────────────────────────────────────────
    best_val_acc = 0.0
    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
        train_loss, train_acc       = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss,   val_acc,  _, _  = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        print(f"Train:      loss={train_loss:.4f}  acc={train_acc:.4f}")
        print(f"Validation: loss={val_loss:.4f}  acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(state, "best_crnn.pth")
            print(f"  -> best model saved (val acc={best_val_acc:.4f})")

    # ── Test ───────────────────────────────────────────────────────────────
    base = model.module if isinstance(model, nn.DataParallel) else model
    base.load_state_dict(torch.load("best_crnn.pth", map_location=device))

    test_loss, test_acc, test_preds, test_labels_out = evaluate(model, test_loader, criterion, device)
    print(f"\nTest loss={test_loss:.4f}  acc={test_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(test_labels_out, test_preds, target_names=class_names))


if __name__ == "__main__":
    main()


Using device: cuda  (2 GPU(s))

ENVIRONMENT CHECK
Python working directory: /home-mscluster/kaspoas/ACML-Project
PyTorch version: 2.5.1+cu121
CUDA available: True
CUDA device count: 2
GPU 0: NVIDIA GeForce GTX 1060 6GB
GPU 1: NVIDIA GeForce GTX 1060 6GB

FMA FILE CHECK
DATA_ROOT: /datasets/kaspoas
tracks.csv path: /datasets/kaspoas/fma_metadata/tracks.csv
tracks.csv exists: True
fma_small directory exists: True
SPEC_DIR: /datasets/kaspoas/mel_spectrograms
mel_spectrograms directory exists: False
tracks.csv shape: (106574, 52)
First 10 metadata columns: [('album', 'comments'), ('album', 'date_created'), ('album', 'date_released'), ('album', 'engineer'), ('album', 'favorites'), ('album', 'id'), ('album', 'information'), ('album', 'listens'), ('album', 'producer'), ('album', 'tags')]
Required column ('set', 'subset') exists: True
Required column ('set', 'split') exists: True
Required column ('track', 'genre_top') exists: True
FMA small rows: 8000
Rows with genre_top: 8000
Split counts:
(s

Train:      loss=2.0117  acc=0.1872
Validation: loss=1.8348  acc=0.3063
  -> best model saved (val acc=0.3063)

Epoch 2/30


Train:      loss=1.9086  acc=0.2442
Validation: loss=1.8120  acc=0.3075
  -> best model saved (val acc=0.3075)

Epoch 3/30


Train:      loss=1.8463  acc=0.2802
Validation: loss=1.7894  acc=0.3262
  -> best model saved (val acc=0.3262)

Epoch 4/30


Train:      loss=1.8228  acc=0.3066
Validation: loss=1.7271  acc=0.3262

Epoch 5/30


...

Train:      loss=1.4294  acc=0.4888
Validation: loss=1.3954  acc=0.5100

Epoch 28/30


Train:      loss=1.4228  acc=0.4997
Validation: loss=1.3975  acc=0.5062

Epoch 29/30


Train:      loss=1.4204  acc=0.4995
Validation: loss=1.3926  acc=0.5288
  -> best model saved (val acc=0.5288)

Epoch 30/30


Train:      loss=1.4096  acc=0.4972
Validation: loss=1.4063  acc=0.5225



Test loss=1.5309  acc=0.4350

Classification Report:
               precision    recall  f1-score   support

   Electronic       0.44      0.61      0.51       100
 Experimental       0.25      0.23      0.24       100
         Folk       0.29      0.20      0.24       100
      Hip-Hop       0.58      0.76      0.66       100
 Instrumental       0.36      0.57      0.44       100
International       0.66      0.40      0.50       100
          Pop       0.23      0.11      0.15       100
         Rock       0.59      0.60      0.60       100

     accuracy                           0.43       800
    macro avg       0.42      0.43      0.42       800
 weighted avg       0.42      0.43      0.42       800

